Dataset Loading
This project aims to predict whether an order item will be returned.

The datasets loaded include:

Orders and Transactions
Product Catalogue
Returns
Customer Purchase and Return History
Marketplace Seller Data
The target variable is Returned:

1 = Returned
0 = Not Returned

In [2]:
import pandas as pd

# Load datasets from your PC
orders = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table2_orders_and_transactions.csv"
)

products = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table3_product_catalogue.csv"
)

returns = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table4_returns.csv"
)

customers = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table5_customer_purchase_and_return_history.csv"
)

sellers = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table6_marketplace_seller_data.csv"
)

# Display dataset dimensions
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Returns:", returns.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)

Orders: (48788, 15)
Products: (3209, 14)
Returns: (7657, 12)
Customers: (48643, 13)
Sellers: (48643, 12)


In [3]:
returned_items = set(
    returns["Order_Item_ID"]
)

orders["Returned"] = (
    orders["Order_Item_ID"]
    .isin(returned_items)
    .astype(int)
)

orders["Returned"].value_counts()

Returned
0    41139
1     7649
Name: count, dtype: int64

Dataset Preprocessing
The datasets are merged using their shared identifiers to create a single modelling dataset.

Data preprocessing includes:

Creating the target variable (Returned)
Joining customer, seller and product information
Handling missing values
Removing duplicate records
Encoding categorical variables
Splitting the data into training, validation and testing datasets
Customer-level splitting is used to prevent data leakage by ensuring that the same customer does not appear in both training and testing data.

In [4]:
print("Orders duplicates:",
      orders.duplicated(subset=["Order_Item_ID"]).sum())

print("Products duplicates:",
      products.duplicated(subset=["SKU"]).sum())

print("Returns duplicates:",
      returns.duplicated(subset=["Return_ID"]).sum())

Orders duplicates: 145
Products duplicates: 9
Returns duplicates: 22


In [5]:
orders = orders.drop_duplicates(
    subset=["Order_Item_ID"]
)

products = products.drop_duplicates(
    subset=["SKU"]
)

returns = returns.drop_duplicates(
    subset=["Return_ID"]
)

In [6]:
df = orders.merge(
    customers,
    on="Order_Item_ID",
    how="left"
)

df = df.merge(
    sellers,
    on="Order_Item_ID",
    how="left"
)

df = df.merge(
    products,
    on="SKU",
    how="left"
)

print(df.shape)

(48643, 52)


In [7]:
missing_values = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(
    missing_values[
        missing_values > 0
    ]
)


Size                      33522
Promotion_Type            29886
Average_Order_Value       16613
Days_Since_Last_Order     16613
Historical_Return_Rate     8878
Colour                     7753
Product_Attributes         5668
Listing_Age                5212
Average_Rating             4375
Review_Count               4138
Product_Subcategory        3586
Brand                      2523
Seller_Return_Rate          419
Average_Shipping_Time       419
Late_Shipment_Rate          419
dtype: int64


In [8]:
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna(
        "Unknown"
    )

In [12]:
from sklearn.model_selection import train_test_split

customer_ids = df["Customer_ID_x"].unique()

train_customers, temp_customers = train_test_split(
    customer_ids,
    test_size=0.30,
    random_state=42
)

val_customers, test_customers = train_test_split(
    temp_customers,
    test_size=0.50,
    random_state=42
)

train_df = df[
    df["Customer_ID_x"].isin(train_customers)
]

val_df = df[
    df["Customer_ID_x"].isin(val_customers)
]

test_df = df[
    df["Customer_ID_x"].isin(test_customers)
]

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (33851, 52)
Validation: (7384, 52)
Testing: (7408, 52)
